In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.font_manager import FontProperties
from skimage import exposure, filters, io, color, morphology, feature, segmentation
from skimage.util import img_as_ubyte, img_as_float
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import glob
import random
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Create necessary directories
def create_directories():
    base_dirs = [
        "dataset/Segmented",
        "dataset/Normalized",
        "dataset/Processed",
        "dataset/Augmented",
        "dataset/Visualization"
    ]
    
    classes = ["Benign", "Early", "Pre", "Pro"]
    
    for base in base_dirs:
        if not os.path.exists(base):
            os.makedirs(base)
            print(f"Created directory: {base}")
        
        # Create class subdirectories
        for cls in classes:
            class_dir = os.path.join(base, cls)
            if not os.path.exists(class_dir):
                os.makedirs(class_dir)
                print(f"Created directory: {class_dir}")
    
    print("Directory structure created successfully.")

In [ ]:
# Preprocessing functions based on Table 12 in the documentation
def normalize_staining(img, target_means=None, target_stds=None):
    """
    Normalize staining using Reinhard method (mentioned in Table 12)
    """
    if target_means is None or target_stds is None:
        # Default values if template not provided
        target_means = np.array([148.60, 160.50, 186.06])
        target_stds = np.array([26.89, 11.32, 9.32])
    
    img = img.astype(np.float32)
    
    # Calculate source statistics
    src_means = np.mean(img, axis=(0, 1))
    src_stds = np.std(img, axis=(0, 1))
    
    # Normalize
    normalized = np.zeros_like(img, dtype=np.float32)
    for i in range(3):
        if src_stds[i] > 0:  # Avoid division by zero
            normalized[:, :, i] = ((img[:, :, i] - src_means[i]) / src_stds[i]) * target_stds[i] + target_means[i]
    
    # Clip values to valid range
    normalized = np.clip(normalized, 0, 255).astype(np.uint8)
    return normalized

In [ ]:
def multi_scale_enhancement(img):
    """
    Multi-scale enhancement using 3-level Gaussian pyramid (mentioned in Table 12)
    """
    # Create a copy to avoid modifying the original
    result = img.copy()
    
    # Create Gaussian pyramid (3-level)
    pyramid = [result]
    for i in range(2):
        result = cv2.pyrDown(result)
        pyramid.append(result)
    
    # Enhance and combine from bottom-up
    enhanced = pyramid[2].copy()  # Start with smallest scale
    for i in range(1, -1, -1):
        # Upscale current enhancement
        enhanced = cv2.pyrUp(enhanced)
        
        # Make sure dimensions match
        h, w = pyramid[i].shape[:2]
        enhanced = cv2.resize(enhanced, (w, h))
            
        # Blend with next level
        enhanced = cv2.addWeighted(enhanced, 0.6, pyramid[i], 0.4, 0)
    
    return enhanced

In [ ]:
def background_removal(img):
    """
    Remove background using adaptive thresholding + watershed (mentioned in Table 12)
    """
    # Convert to grayscale if it's not
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img.copy()
    
    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                cv2.THRESH_BINARY_INV, 11, 2)
    
    # Noise removal with morphological operations
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Sure background area
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    
    # Finding sure foreground area
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    ret, sure_fg = cv2.threshold(dist_transform, 0.5*dist_transform.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)
    
    # Finding unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # Marker labelling
    ret, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    # Apply watershed
    if len(img.shape) == 3:
        markers = cv2.watershed(img, markers)
    else:
        # Convert gray to BGR for watershed
        img_color = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        markers = cv2.watershed(img_color, markers)
    
    # Create mask
    mask = np.zeros_like(gray)
    mask[markers > 1] = 255
    
    # Apply mask to original image
    if len(img.shape) == 3:
        result = np.zeros_like(img)
        for i in range(3):
            result[:, :, i] = cv2.bitwise_and(img[:, :, i], mask)
    else:
        result = cv2.bitwise_and(gray, mask)
    
    return result

In [ ]:
def edge_enhancement(img):
    """
    Enhance edges using Laplacian of Gaussian (sigma=1.5, mentioned in Table 12)
    """
    # Convert to grayscale if needed
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img.copy()
    
    # Apply Laplacian of Gaussian
    blur = cv2.GaussianBlur(gray, (5, 5), 1.5)
    laplacian = cv2.Laplacian(blur, cv2.CV_8U, ksize=3)
    
    # Add back to original image
    if len(img.shape) == 3:
        result = img.copy()
        for i in range(3):
            result[:, :, i] = cv2.addWeighted(img[:, :, i], 1, laplacian, 0.5, 0)
    else:
        result = cv2.addWeighted(gray, 1, laplacian, 0.5, 0)
    
    return result

In [ ]:
def contrast_stretching(img):
    """
    Apply CLAHE (clip=2.0, tile=8×8, mentioned in Table 12)
    """
    # Create CLAHE object
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    
    if len(img.shape) == 3:
        # Convert to LAB color space
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        
        # Apply CLAHE to L channel
        l, a, b = cv2.split(lab)
        cl = clahe.apply(l)
        
        # Merge channels
        merged = cv2.merge((cl, a, b))
        
        # Convert back to RGB
        result = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    else:
        # Apply CLAHE directly to grayscale
        result = clahe.apply(img)
    
    return result

In [ ]:
def noise_reduction(img):
    """
    Apply Non-local means (h=10, patchSize=7, mentioned in Table 12)
    """
    # Parameters
    h = 10  # Filter strength
    template_window_size = 7  # Size of the template patch
    search_window_size = 21  # Size of the window for searching similar patches
    
    # Apply non-local means denoising
    if len(img.shape) == 3:
        result = cv2.fastNlMeansDenoisingColored(
            img, None, h, h, template_window_size, search_window_size)
    else:
        result = cv2.fastNlMeansDenoising(
            img, None, h, template_window_size, search_window_size)
    
    return result

In [ ]:
def augment_image(image):
    """Apply random augmentations to an image"""
    img = image.copy()
    augmentation_type = np.random.choice(['rotate', 'flip', 'brightness', 'combined'])
    
    if augmentation_type == 'rotate' or augmentation_type == 'combined':
        # Random rotation
        angle = np.random.uniform(-20, 20)
        height, width = img.shape[:2]
        center = (width // 2, height // 2)
        rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        img = cv2.warpAffine(img, rotation_matrix, (width, height), 
                             borderMode=cv2.BORDER_REFLECT)
    
    if augmentation_type == 'flip' or augmentation_type == 'combined':
        # Random flip
        flip_code = np.random.choice([-1, 0, 1])
        img = cv2.flip(img, flip_code)
    
    if augmentation_type == 'brightness' or augmentation_type == 'combined':
        # Random brightness and contrast
        alpha = 1.0 + np.random.uniform(-0.2, 0.2)  # Contrast control
        beta = np.random.uniform(-20, 20)  # Brightness control
        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    
    return img

In [ ]:
class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(AttentionUNet, self).__init__()
        
        # Encoder (downsampling)
        self.enc1 = self.conv_block(in_channels, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.enc4 = self.conv_block(256, 512)
        
        # Bridge
        self.bridge = self.conv_block(512, 1024)
        
        # Decoder (upsampling) with attention
        self.upconv1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.attention1 = self.attention_block(512, 512)
        self.dec1 = self.conv_block(1024, 512)
        
        self.upconv2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.attention2 = self.attention_block(256, 256)
        self.dec2 = self.conv_block(512, 256)
        
        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.attention3 = self.attention_block(128, 128)
        self.dec3 = self.conv_block(256, 128)
        
        self.upconv4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.attention4 = self.attention_block(64, 64)
        self.dec4 = self.conv_block(128, 64)
        
        # Output layer
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
        
        # Max pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def attention_block(self, g_channels, x_channels):
        """Attention gate"""
        inter_channels = x_channels // 4
        
        # Gating signal processing (g -> W_g)
        Wg = nn.Sequential(
            nn.Conv2d(g_channels, inter_channels, kernel_size=1),
            nn.BatchNorm2d(inter_channels)
        )
        
        # Input signal processing (x -> W_x)
        Wx = nn.Sequential(
            nn.Conv2d(x_channels, inter_channels, kernel_size=1),
            nn.BatchNorm2d(inter_channels)
        )
        
        # Attention mechanism
        psi = nn.Sequential(
            nn.Conv2d(inter_channels, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        # ReLU activation
        relu = nn.ReLU(inplace=True)
        
        # Return the attention calculation as a function
        def forward(g, x):
            g1 = Wg(g)
            x1 = Wx(x)
            # Element-wise addition followed by ReLU
            out = relu(g1 + x1)
            # Element-wise multiplication with input after attention map
            attention = psi(out)
            out = x * attention
            return out
            
        return forward
    
    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))
        
        # Bridge
        bridge = self.bridge(self.pool(enc4))
        
        # Decoder with attention and skip connections
        upconv1 = self.upconv1(bridge)
        attention1 = self.attention1(upconv1, enc4)
        dec1 = self.dec1(torch.cat([attention1, upconv1], dim=1))
        
        upconv2 = self.upconv2(dec1)
        attention2 = self.attention2(upconv2, enc3)
        dec2 = self.dec2(torch.cat([attention2, upconv2], dim=1))
        
        upconv3 = self.upconv3(dec2)
        attention3 = self.attention3(upconv3, enc2)
        dec3 = self.dec3(torch.cat([attention3, upconv3], dim=1))
        
        upconv4 = self.upconv4(dec3)
        attention4 = self.attention4(upconv4, enc1)
        dec4 = self.dec4(torch.cat([attention4, upconv4], dim=1))
        
        # Output layer
        out = self.outc(dec4)
        return self.sigmoid(out)

In [ ]:
def segment_cells(img, model):
    """
    Segment cells using the U-Net with attention model
    For demonstration, we'll use a simple thresholding approach instead of the deep learning model
    as training would be required for the actual model
    """
    # In a real scenario, you would use the trained model
    # For demonstration, we'll simulate it with classical image processing
    
    # Convert to grayscale if it's not
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img.copy()
    
    # Apply Otsu's thresholding
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    
    # Morphological operations to remove small noise
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Dilate to ensure cells are fully covered
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    
    # Apply distance transform to find centers of cells
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist_transform, 0.5*dist_transform.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)
    
    # Find unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # Marker labelling
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    # Apply watershed
    if len(img.shape) == 3:
        markers = cv2.watershed(img, markers)
    else:
        img_color = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        markers = cv2.watershed(img_color, markers)
    
    # Create segmentation mask
    segmentation_mask = np.zeros_like(gray)
    segmentation_mask[markers > 1] = 255
    
    # Apply mask to original image
    if len(img.shape) == 3:
        segmented = np.zeros_like(img)
        for i in range(3):
            segmented[:, :, i] = cv2.bitwise_and(img[:, :, i], segmentation_mask)
    else:
        segmented = cv2.bitwise_and(gray, segmentation_mask)
    
    return segmented, segmentation_mask

In [ ]:
def process_and_segment_images():
    # Load template image for normalization
    template_path = "dataset/normalized_template.jpg"
    try:
        template_img = cv2.imread(template_path)
        if template_img is None:
            print(f"Template image not found or couldn't be loaded: {template_path}")
            print("Using default normalization values.")
            template_means = np.array([148.60, 160.50, 186.06])
            template_stds = np.array([26.89, 11.32, 9.32])
        else:
            template_img = cv2.cvtColor(template_img, cv2.COLOR_BGR2RGB)
            template_means = np.mean(template_img, axis=(0, 1))
            template_stds = np.std(template_img, axis=(0, 1))
            print(f"Template image loaded successfully. Mean: {template_means}, Std: {template_stds}")
    except Exception as e:
        print(f"Error loading template image: {e}")
        template_means = np.array([148.60, 160.50, 186.06])
        template_stds = np.array([26.89, 11.32, 9.32])
    
    # Initialize a dummy model (in a real scenario, you would load a trained model)
    # For demonstration, we'll use the classical segmentation approach
    model = None
    
    # Define classes
    classes = ["Benign", "Early", "Pre", "Pro"]
    
    # Process images for each class
    for cls in classes:
        # Check if the source directory exists
        source_dir = os.path.join("dataset/Original", cls)
        if not os.path.exists(source_dir):
            print(f"Source directory not found: {source_dir}")
            continue
        
        # Get list of images
        image_files = glob.glob(os.path.join(source_dir, "*.jpg"))
        if not image_files:
            print(f"No images found in {source_dir}")
            continue
        
        print(f"Processing {len(image_files)} images for class {cls}")
        
        # Process images
        for img_path in tqdm(image_files, desc=f"Processing {cls}"):
            # Read image
            img = cv2.imread(img_path)
            if img is None:
                print(f"Could not read image: {img_path}")
                continue
            
            # Convert to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Get image filename without extension
            file_name = os.path.basename(img_path)
            
            # Apply preprocessing steps
            
            # 1. Stain Normalization
            normalized = normalize_staining(img, template_means, template_stds)
            # Save normalized image
            cv2.imwrite(os.path.join("dataset/Normalized", cls, file_name), 
                        cv2.cvtColor(normalized, cv2.COLOR_RGB2BGR))
            
            # 2. Multi-scale Enhancement
            enhanced = multi_scale_enhancement(normalized)
            
            # 3. Background Removal
            bg_removed = background_removal(enhanced)
            
            # 4. Edge Enhancement
            edge_enhanced = edge_enhancement(bg_removed)
            
            # 5. Contrast Stretching
            contrast_stretched = contrast_stretching(edge_enhanced)
            
            # 6. Noise Reduction
            denoised = noise_reduction(contrast_stretched)
            
            # Save processed image
            cv2.imwrite(os.path.join("dataset/Processed", cls, file_name), 
                        cv2.cvtColor(denoised, cv2.COLOR_RGB2BGR))
            
            # 7. Cell Segmentation
            segmented, mask = segment_cells(denoised, model)
            
            # Save segmented image
            cv2.imwrite(os.path.join("dataset/Segmented", cls, file_name), 
                        cv2.cvtColor(segmented, cv2.COLOR_RGB2BGR))
            
            # Create augmented versions (3 variations)
            for i in range(3):
                augmented = augment_image(denoised)
                aug_filename = f"aug{i+1}_{file_name}"
                cv2.imwrite(os.path.join("dataset/Augmented", cls, aug_filename), 
                            cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))
    
    print("Processing completed successfully.")

In [ ]:
def create_processing_visualization():
    """Create visualization grid showing original images and processing steps"""
    classes = ["Benign", "Early", "Pre", "Pro"]
    
    # For each class, select one sample image
    plt.figure(figsize=(20, 16))
    
    # Set font properties
    title_font = FontProperties(family='Times New Roman', size=22)
    subtitle_font = FontProperties(family='Times New Roman', size=18)
    
    for i, cls in enumerate(classes):
        # Find a sample image
        source_dir = os.path.join("dataset/Original", cls)
        image_files = glob.glob(os.path.join(source_dir, "*.jpg"))
        
        if not image_files:
            print(f"No images found for class {cls}")
            continue
        
        # Select first image
        img_path = image_files[0]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Get image filename
        file_name = os.path.basename(img_path)
        
        # Load other processed versions
        normalized_path = os.path.join("dataset/Normalized", cls, file_name)
        processed_path = os.path.join("dataset/Processed", cls, file_name)
        segmented_path = os.path.join("dataset/Segmented", cls, file_name)
        augmented_path = os.path.join("dataset/Augmented", cls, f"aug1_{file_name}")
        
        normalized = cv2.imread(normalized_path)
        normalized = cv2.cvtColor(normalized, cv2.COLOR_BGR2RGB) if normalized is not None else None
        
        processed = cv2.imread(processed_path)
        processed = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB) if processed is not None else None
        
        segmented = cv2.imread(segmented_path)
        segmented = cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB) if segmented is not None else None
        
        augmented = cv2.imread(augmented_path)
        augmented = cv2.cvtColor(augmented, cv2.COLOR_BGR2RGB) if augmented is not None else None
        
        # Plot row for this class
        row = i + 1
        
        # Original image
        plt.subplot(4, 5, 5*i + 1)
        plt.imshow(img)
        plt.title(f"{cls} - Original", fontproperties=subtitle_font)
        plt.axis('off')
        
        # Normalized image
        if normalized is not None:
            plt.subplot(4, 5, 5*i + 2)
            plt.imshow(normalized)
            plt.title("Normalized", fontproperties=subtitle_font)
            plt.axis('off')
        
        # Processed image
        if processed is not None:
            plt.subplot(4, 5, 5*i + 3)
            plt.imshow(processed)
            plt.title("Processed", fontproperties=subtitle_font)
            plt.axis('off')
        
        # Segmented image
        if segmented is not None:
            plt.subplot(4, 5, 5*i + 4)
            plt.imshow(segmented)
            plt.title("Segmented", fontproperties=subtitle_font)
            plt.axis('off')
        
        # Augmented image
        if augmented is not None:
            plt.subplot(4, 5, 5*i + 5)
            plt.imshow(augmented)
            plt.title("Augmented", fontproperties=subtitle_font)
            plt.axis('off')
    
    plt.suptitle("Leukemia Classification: Image Processing Pipeline", fontproperties=title_font)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    
    # Save the visualization
    plt.savefig(os.path.join("dataset/Visualization", "processing_visualization.jpg"), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join("dataset/Visualization", "processing_visualization.pdf"), dpi=1000, bbox_inches='tight')
    print("Created processing visualization.")

In [ ]:
def create_segmentation_visualization():
    """Create visualization showing original images and their segmentation"""
    classes = ["Benign", "Early", "Pre", "Pro"]
    
    plt.figure(figsize=(16, 12))
    
    # Set font properties
    title_font = FontProperties(family='Times New Roman', size=22)
    subtitle_font = FontProperties(family='Times New Roman', size=18)
    
    for i, cls in enumerate(classes):
        # Find a sample image
        source_dir = os.path.join("dataset/Original", cls)
        image_files = glob.glob(os.path.join(source_dir, "*.jpg"))
        
        if not image_files:
            print(f"No images found for class {cls}")
            continue
        
        # Select first image
        img_path = image_files[0]
        file_name = os.path.basename(img_path)
        
        # Original image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Segmented image
        segmented_path = os.path.join("dataset/Segmented", cls, file_name)
        segmented = cv2.imread(segmented_path)
        
        if segmented is not None:
            segmented = cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB)
            
            # Display original in top row
            plt.subplot(2, 4, i+1)
            plt.imshow(img)
            plt.title(f"{cls} - Original", fontproperties=subtitle_font)
            plt.axis('off')
            
            # Display segmented in bottom row
            plt.subplot(2, 4, i+5)
            plt.imshow(segmented)
            plt.title(f"{cls} - Segmented", fontproperties=subtitle_font)
            plt.axis('off')
    
    plt.suptitle("Leukemia Classification: Cell Segmentation Results", fontproperties=title_font)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    
    # Save the visualization
    plt.savefig(os.path.join("dataset/Visualization", "segmentation_visualization.jpg"), dpi=300, bbox_inches='tight')
    print("Created segmentation visualization.")

In [ ]:
if __name__ == "__main__":
    print("Starting leukemia image preprocessing pipeline...")
    
    # Create directory structure
    create_directories()
    
    # Process and segment all images
    process_and_segment_images()
    
    # Create visualizations
    create_processing_visualization()
    create_segmentation_visualization()
    
    print("Image preprocessing pipeline completed successfully.")